# Campaign Automation with n8n — Analysis Notebook

**Part 2 of the dunnhumby retail intelligence series**
Part 1: Retail Store Performance Analysis (DiD) → **Part 2: this notebook** → Part 3: decision-intelligence agent (v4).

> ## Executive summary — read this first
>
> This notebook makes the automation system's business logic **inspectable**: every rule that
> `main.py` enforces in the FastAPI service is reproduced here in pandas, so a reviewer can see
> *why* a store or customer was selected without reading the server code.
>
> | Decision | Rule (enforced in `main.py`) | Source |
> |---|---|---|
> | Eligible stores | 85 stores, ranked by Best-Customer count | Part-1 RFM segmentation |
> | Phase slices | Pilot = top 5 · Phase 1 = next 25 · Phase 2 = next 55 | Phased rollout design |
> | Target customers | `segment_cust == "Best Customers"` | Part-1 RFM (causally validated: +2.84% ITT) |
> | Timing | 12 PM – 6 PM | Part-1 engagement analysis |
> | Rollout gate | Pilot uplift CI low > 0 → ADVANCE, else HOLD | `validate_campaign_benchmark()` |
>
> **Causal honesty:** the forecast benchmark (+30.1%) absorbs +9.7% market drift. The causal DiD
> on the same window is **+2.84% household ITT** (p=0.10) / **−9.6% store-level**. Part 3's agent
> gates scale-up on the **causal 3% target** — set `PILOT_UPLIFT_PCT=2.84` (with the CI env
> overrides) in the service to align this system's gate with the causal estimate.


## 1 · Setup

Load the same static datasets the FastAPI service uses (served from the repo via `STORE_DATA_URL`
/ `CUSTOMER_DATA_URL` in `main.py`; read locally here so the notebook runs offline).


In [ ]:
# ---- Configuration (mirrors main.py constants) -------------------------------
from pathlib import Path
import json

import pandas as pd

DATA_DIR = Path("datasets")
AUDIT_LOG_PATH = Path("audit_log.jsonl")
OUTPUT_DIR = Path("outputs")

# Phased rollout slices from filter_stores_by_phase() in main.py
PHASE_SLICES = {"Pilot": (0, 5), "Phase 1": (5, 30), "Phase 2": (30, 85)}
TARGET_SEGMENT = "Best Customers"
TIMING_WINDOW = "12 PM - 6 PM"


In [ ]:
# ---- Load stores and customers (same files the API serves) -------------------
stores = pd.read_csv(DATA_DIR / "stores.csv")
stores.columns = stores.columns.str.strip().str.lower()

customers = pd.read_csv(DATA_DIR / "customer demographic.csv")
customers.columns = customers.columns.str.strip().str.lower()

print(f"stores: {len(stores)} | customers: {len(customers)}")
stores.head()


## 2 · Store eligibility & phase selection

`filter_stores_by_phase()` ranks eligible stores by **Best-Customer count** (from Part-1's RFM
segmentation — not a re-derived total-customer threshold) and slices the ranked list per phase.
The 426 non-eligible "zombie stores" are excluded upstream: they have too few loyalty households
to respond to any campaign (dunnhumby's personalised-offers methodology: targeting without an
addressable customer base is wasted spend).


In [ ]:
# ---- Replicate filter_stores_by_phase() from main.py -------------------------
def stores_for_phase(stores: pd.DataFrame, phase: str) -> pd.DataFrame:
    """Rank by Best-Customer count and slice the phase window."""
    lo, hi = PHASE_SLICES[phase]
    ranked = stores.sort_values(by="total_customer", ascending=False)
    return ranked.iloc[lo:hi].copy()


for phase in PHASE_SLICES:
    sel = stores_for_phase(stores, phase)
    print(f"{phase}: {len(sel)} stores | best-customer range "
          f"{sel['total_customer'].min()}–{sel['total_customer'].max()}")

pilot = stores_for_phase(stores, "Pilot")
pilot[["store_id", "total_customer"]]


## 3 · Customer targeting

`select_target_customers()` targets households by their **actual RFM segment label** rather than
reconstructing "Best Customers" from demographic proxies — the label carries the Part-1
behavioural evidence, and Part-1's DiD confirmed the segment responds (+2.84% ITT). Demographics
(45–54, $50–74K, 2 adults no kids) are used for creative personalisation, not for eligibility.


In [ ]:
# ---- Replicate select_target_customers() + email generation ------------------
def target_customers(customers: pd.DataFrame) -> pd.DataFrame:
    """Keep only the RFM Best Customers segment; attach placeholder emails."""
    best = customers[customers["segment_cust"] == TARGET_SEGMENT].copy()
    best["email"] = best["household_key"].astype(str) + "@campaign18.com"
    return best


targets = target_customers(customers)
print(f"customers targeted: {len(targets)} ({len(targets) / len(customers):.1%} of households)")
# NOTE (main.py KNOWN LIMITATION): emails are synthetic placeholders — the dataset
# has no contact field. test_mode=True must stay on until a real CRM source exists.
targets.head()


## 4 · Campaign execution & audit trail

The n8n workflow calls `GET /run-campaign` on a schedule. The service selects stores/customers
using the rules above, optionally checks the live forecast signal per store, writes one
timestamped audit record, and (if `test_mode=false`) creates the campaign in Brevo.
Every record ends up in `audit_log.jsonl` — the single source of truth for Part 3's agent.


In [ ]:
# ---- Inspect the real audit log produced by the deployed service -------------
runs = [json.loads(line) for line in AUDIT_LOG_PATH.read_text().splitlines() if line.strip()]
print(f"audit runs recorded: {len(runs)}")
pd.DataFrame(runs)[["run_timestamp", "phase", "stores_selected", "store_ids",
                    "customers_targeted", "rollout_decision"]].head()


In [ ]:
# ---- Visual evidence of the automated run (from outputs/) --------------------
from IPython.display import Image, display

for name in ("n8n workflow execution.jpg", "api response.jpg", "stakeholder email.jpg"):
    print(f"--- {name} ---")
    display(Image(str(OUTPUT_DIR / name), width=640))


## 5 · Rollout decision gate — forecast vs causal

`validate_campaign_benchmark()` advances the phase when the pilot uplift CI's lower bound is
above zero. Two benchmarks exist and must not be conflated:

| Benchmark | Value | 95% CI | Use |
|---|---|---|---|
| Forecast (LightGBM counterfactual) | **+30.1%** | [+11.9%, +51.0%] | Default; absorbs +9.7% market drift |
| Causal (household DiD, ITT) | **+2.84%** | [−0.5%, +6.2%] | **Set via env to gate causally** |

**Fixed in this revision:** the gate is env-configurable (`PILOT_UPLIFT_PCT` / `PILOT_CI_LOW` /
`PILOT_CI_HIGH`; default = forecast figure with an explicit origin label in
`validation_reason`). Part 3's decision agent reads the audit log and re-gates every
recommendation against the **3% causal target** with matched-control DiD — so a wrong phase
advance here cannot silently propagate into scale-up.


## 6 · System architecture & known limitations

| Layer | Tool | Role |
|---|---|---|
| Business logic | FastAPI (Render) | Store scoring, eligibility, targeting rules |
| Orchestration | n8n | Scheduled trigger, phase advancement, failure handling |
| Delivery | Brevo | Email campaign execution |
| Audit | JSONL / Google Sheets | Timestamped record per run |
| Stakeholder alerts | Gmail | Deployment summary per phase |

**Known limitations** (also in README): in-memory phase state resets on restart; synthetic
emails (keep `test_mode=true`); no automatic retry; benchmark defaults to the forecast figure.

**Handoff to Part 3:** the audit contract endpoint (`GET /audit?schema=contract`) exposes only
`campaign / timing / run_timestamp / store_ids`. The v4 agent consumes exactly that contract,
re-evaluates every store with the causal guardrail, and holds scale-ups for human approval.


## References

1. dunnhumby — *Personalised Offers for Retailers* (campaign framework, addressable-base targeting)
2. dunnhumby — *Retail:Vision* (2026) — connected AI-powered decision making (series motivation)
3. WARC — *ROI of Successful Campaigns Continues to Grow* (ROI benchmark, Part-1 deck)
4. Part-1 notebook, DiD Validation section — causal estimates (+2.84% ITT / −9.6% store) and the $14K revised impact
5. Part-3 repo — `retail-decision-intelligence-agent-v4` (`decision_engine/calibration.py:TARGET_UPLIFT_PCT = 3.0`)
